In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.functions import to_date, date_format

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta
from statsmodels.tsa.seasonal import STL
from sklearn.linear_model import LinearRegression
from scipy.stats import ttest_ind


In [ ]:
# Check if the enviornment allows CausalImpact to be installed
try:
    from causalimpact import CausalImpact
    HAS_CAUSALIMPACT = True
except Exception:
    HAS_CAUSALIMPACT = False
    print("Failed to install CausalImpact!")


import warnings
warnings.filterwarnings("ignore")

### Load Retailer Cardholder Data


In [ ]:
# Import the data model, join tables
df_sales = spark.table("sales_data")
df_competitors = spark.table("competitor_sales_data")
df_dim_hier_retailer = spark.table("product_hierarchy_retailer")
df_dim_hier_secondary = spark.table("product_hierarchy_secondary")
df_dim_product = spark.table("product_dimension")
df_dim_store = spark.table("store_dimension")
df_dim_country = spark.table("country_dimension")
df_dim_customer = spark.table("customer_dimension")
df_dim_supplier = spark.table("supplier_dimension")
df_dim_businessbrand = spark.table("business_brand_dimension")
df_dim_calendar = spark.table("calendar_dimension")
df_dim_channel = spark.table("channel_dimension")


In [ ]:
#Joins

df = (df_competitors
      .join(df_dim_hier_retailer, ['Retailer_prodhier_id','barcode'], "inner")
      .join(df_dim_store, ['store_id'], "inner")
      .join(df_dim_country, ['country_id'], "inner")
      .join(df_dim_channel, ['channel_key'], 'inner')
      .join(df_dim_product.drop('barcode','customer_id','supplier_id'), F.col('cust_product_id') == F.col('product_id'), "inner")
      .join(df_dim_customer, ['customer_id'], "inner")
      .join(df_dim_supplier, ['supplier_id'], "inner")
      .join(df_dim_businessbrand, ['Business_brand_key'], "inner")
      .join(df_dim_calendar, ['period_date'], "inner")
      #.where(F.col('country_name') == "France")
)


In [ ]:
df.printSchema()

In [ ]:
# Convert to date and extract day of week
df = df.withColumn("day_of_week", date_format(to_date("period_date"), "EEEE"))

### Apply filters

In [ ]:
# 1. Filter for France
df_FR = df.filter(col('country_name') == 'France')

In [ ]:
# 2. Filter for yoghurt categories - checkpoint
df_FR_yoghurt = df_FR.filter(col('class_desc').isin('FLAVOURED/FRUIT YOGURTS','DESSERTS','FRESH CHEESE','PLAIN YOGURTS','HEALTH/ORGANIC YOGURTS','DRINKING YOGURTS/SNACKS'))


In [ ]:
# 3. Filter for "in-store" purchases
df_FR_yoghurt_instore = df_FR_yoghurt.filter(col('sub_channel_type') == "In Store")

In [ ]:
# 4. Filter for the top 100 stores by total sales

df_top100_FR_yoghurt_instore = (df_FR_yoghurt_instore.groupBy("store_id", "country_name","store_city","business_brand","store_address_1")
      .agg(F.sum("total_sales_amount").alias("total_sales_amount"), 
           F.min(F.to_date("period_date")).alias("min_date"),
           F.max(F.to_date("period_date")).alias("max_date"))
      .orderBy(col("total_sales_amount").desc())
      .limit(100)
)

df_FR_yoghurt_instore_top100 = df_FR_yoghurt_instore.join(df_top100_FR_yoghurt_instore.select('store_id'), ['store_id'], 'inner')

In [ ]:
# Checkpoint
df_FR_yoghurt_instore_top100.agg(F.countDistinct('barcode'), F.sum('total_sales_amount')).show()

In [ ]:
df_FR_yoghurt_instore_top100.display()

### Preprocessing

#### Aggregate SKUs/Barcodes

In [ ]:
# Import SKU info from the France sell-out extract
skuinfo_path = './data/sellout_skus_france.csv'

df_skuinfo_sellout = spark.createDataFrame(pd.read_csv(skuinfo_path, encoding= 'utf-8')) \
  .withColumn("EAN_CODE_MDM", col("EAN_CODE_MDM").cast("string"))

# Join SKU info
df_FR_yoghurt_instore_top100_2 = df_FR_yoghurt_instore_top100.join(df_skuinfo_sellout, 
                                                                   df_FR_yoghurt_instore_top100['barcode'] == df_skuinfo_sellout['EAN_CODE_MDM'],how='left') \
  .withColumn('flag_Lot1b', F.when(col("EAN_CODE_MDM").isNotNull(), 1).otherwise(0))


In [ ]:
df_FR_yoghurt_instore_top100_2.display()

In [ ]:
df_FR_yoghurt_instore_top100_2.printSchema()

In [ ]:
# Checkpoint
df_FR_yoghurt_instore_top100_2.groupBy('flag_Lot1b') \
  .agg(F.countDistinct('barcode'), F.sum('total_sales_amount')).show()

In [ ]:
# Create the column containing the Analytical SKU (level at which the SKU's are being analyzed) : supplier x brand
# Examples: SUPPLIER_A_BRAND_1_YOGURT, SUPPLIER_A_BRAND_2_DESSERT, SUPPLIER_B_BRAND_1_YOGURT

df_FR_yoghurt_instore_top100_3 = (df_FR_yoghurt_instore_top100_2
  .withColumn('TYPE_OF_YOGHURT_2', F.when(col('EAN_CODE_MDM').isNotNull(), col('GL_TYPE_OF_YOGHURT')).otherwise(F.lit("OTHER")))
  .withColumn('analytical_SKU',   
              F.concat_ws("_", 
                          F.trim(F.coalesce(F.col("Supplier_holding_desc"), F.lit("UnknownManufacturer"))),
                          F.trim(F.coalesce(F.col("Brand_desc"), F.lit("UnknownBrand"))),
                          F.trim(F.coalesce(F.col("TYPE_OF_YOGHURT_2"), F.lit("OTHER")))
                          #F.trim(F.coalesce(F.col("PACKSIZE_MDM"), F.lit("OTHER")))
                          )))


#### Analytical Dataframe

In [ ]:
df_FR_yoghurt_instore_top100_3.printSchema()

In [ ]:
df_FR_yoghurt_instore_top100_3.display()

In [ ]:
# Define categorical and numerical columns
categorical_cols = ['period_date', 'department_desc', 'Grp_class_Desc',
                    'Supplier_holding_desc','Brand_desc', 
                    'store_key','region_name', 'store_city', 'store_description', 'store_address_1','country_name', 'channel_type', 'sub_channel_type', 'business_brand', 'business_service',
                    'analytical_SKU'] # Categorical columns to group by # Removed class_desc (as it was leading to duplicate anlaytical SKUs)

numerical_cols = ['total_sales_amount', 'total_promo_sales_amount', 'total_qty', 'total_qty_promo', 'number_of_transactions'] # Columns to aggregate, numerical values

In [ ]:
# Aggregate to analytical product level (i.e. Manufacturer x Brand x Type of Yoghurt)
# Note: Creates column 'barcode_itemdesc_pairs' to collect individual SKUs and their item descriptions that have been aggregated under one Analytical SKU
# eg. (barcode_itemdesc_pairs) " "
df_cannibalisation = (df_FR_yoghurt_instore_top100_3.select(*(categorical_cols + numerical_cols + ['barcode', 'item_desc', 'class_desc']))
                      .groupBy(*(categorical_cols))
                      .agg(F.sum('total_sales_amount').alias('total_sales_amount'),
                           F.sum('total_promo_sales_amount').alias('total_promo_sales_amount'),
                           F.sum('total_qty').alias('total_qty'),
                           F.sum('total_qty_promo').alias('total_qty_promo'),
                           F.sum('number_of_transactions').alias('number_of_transactions'),
                           F.collect_set(F.array('barcode', 'item_desc', 'class_desc')).alias('barcode_itemdesc_pairs'))
                      )

In [ ]:
# Save as temporary table
df_cannibalisation.createOrReplaceTempView("df_cannibalisation")

In [ ]:
display(df_cannibalisation)

### Filter data for top 25 in France


In [ ]:
df_top20 = df_cannibalisation.groupBy("store_key").agg(F.sum("total_sales_amount").alias("total_sales_amount")).orderBy(F.col("total_sales_amount").desc()).limit(20)

In [ ]:
display(df_top20)

In [ ]:
top20_store_ids = [row["store_key"] for row in df_top20.collect()]

In [ ]:
#Filter for top 20 stores
dfcan_top20 = df_cannibalisation.filter(col("store_key").isin(top20_store_ids))

In [ ]:
#Top 20 stores to Pandas
dfPdcan_top20 = dfcan_top20.toPandas()

In [ ]:
dfPdcan_top20 = dfPdcan_top20.drop(columns = ["number_of_transactions", "business_service", "store_address_1", "store_city", "sub_channel_type", "country_name", "channel_type", "region_name"])

In [ ]:
dfPdcan_top20.info()

In [ ]:
dfcan_top20.select("store_description").distinct().collect()

In [ ]:
dfPdcan_top20["analytical_SKU"].nunique()

### Modelling / Pipeline Functions



#### Utility functions

In [ ]:
# UTILITIES

def _stl_decompose(series: pd.Series, period: int = 7):
    """
    STL decomposition returning (trend, seasonal, resid).
    """
    series = pd.Series(series).astype(float).fillna(0.0)
    stl = STL(series, period=period, robust=True)
    res = stl.fit()
    return res.trend, res.seasonal, res.resid

#STL decompose is used to decompose time series into trend, seasonal and resid, in order to distinguish between repetitive patterns, for example weekends, and a true trend because of the promos

#### Preprocessing

In [ ]:
# =========================
# Step 1: Preprocessing
# =========================

def build_core_matrices(df: pd.DataFrame, sku_col: str = "analytical_SKU", date_col: str = "period_date", qty_col: str = "total_qty", 
                        promo_qty_col: str = "total_qty_promo", dept_col: str = "Grp_class_Desc"):
    """
    Builds three matrices :
      Y: [date x SKU] sales (total_qty)
      P: [date x SKU] boolean promo indicator (total_qty_promo > 0)
      D: [date x department] total daily sales within the entire department
      meta: { 'sku_dept': mapping SKU -> department, 'date_index': index }
    Only uses required columns per your spec.
    """
    cols_needed = [date_col, sku_col, dept_col, qty_col, promo_qty_col]
    df = df[cols_needed].copy()
    df[date_col] = pd.to_datetime(df[date_col]).dt.normalize()
    df["qty"] = pd.to_numeric(df[qty_col], errors="coerce").fillna(0.0)
    df["promo_qty"] = pd.to_numeric(df[promo_qty_col], errors="coerce").fillna(0.0)
    df["promo_flag"] = (df["promo_qty"] > 0).astype(int)

    # Y and P
    Y = (df.pivot_table(index=date_col, columns=sku_col, values="qty", aggfunc="sum").sort_index().fillna(0.0))
    P = (df.pivot_table(index=date_col, columns=sku_col, values="promo_flag", aggfunc="max").sort_index().fillna(0).astype(int))

    # D (department totals)
    D = (df.groupby([date_col, dept_col])["qty"].sum().reset_index().pivot(index=date_col, columns=dept_col, values="qty").sort_index().fillna(0.0))

    # Create the mapping of SKU -> department (i.e. Grp_class_Desc) (choose most frequent mapping)
    sku_dept = (df.groupby([sku_col, dept_col]).size().reset_index(name="n").sort_values([sku_col, "n"], ascending=[True, False]))
    sku_dept = sku_dept.drop_duplicates(subset=[sku_col]).set_index(sku_col)[dept_col].to_dict()

    meta = {"sku_dept": sku_dept, "date_index": Y.index}
    return Y, P, D, meta


def decompose_skus_and_availability(Y: pd.DataFrame, weekly_period: int = 7, availability_threshold: float = 0.5):
    """
    For each SKU: STL decomposition of Y -> trend/seasonal/residual.
    Build availability A: A[t, sku] = 1 if trend >= threshold else 0.
    """
    trend, seasonal, resid = {}, {}, {}
    A_cols = []
    for sku in Y.columns:
        s = Y[sku].astype(float).fillna(0.0)
        tr, se, re = _stl_decompose(s, period=weekly_period)
        trend[sku], seasonal[sku], resid[sku] = tr, se, re
        A_cols.append((tr >= availability_threshold).astype(int).rename(sku))
    A = pd.concat(A_cols, axis=1)
    return trend, seasonal, resid, A


def decompose_department_signals(D: pd.DataFrame, weekly_period: int = 7):
    """
    Decompose department totals to get:
      Q_trend: proxy for store/department performance (trend)
      Q_seasonal: weekly seasonality per department
    """
    Q_trend = pd.DataFrame(index=D.index)
    Q_seasonal = pd.DataFrame(index=D.index)
    for dept in D.columns:
        s = D[dept].astype(float).fillna(0.0)
        tr, se, _ = _stl_decompose(s, period=weekly_period)
        Q_trend[dept] = tr
        Q_seasonal[dept] = se
    return Q_trend, Q_seasonal

#### Selecting innovations and Potential Cannibals

In [ ]:
# =========================
# Step 2: Innovation-based cannibalization
# =========================

def detect_innovation_launches(
    Y: pd.DataFrame,
    P: pd.DataFrame,
    A: pd.DataFrame,
    sku2dept: dict,
    lookback_days: int = 365,
    min_launch_sales: float = 3.0,
    min_post_days: int = 14,
    use_availability_filter: bool = True
):
    """
    Detect SKUs that behave like innovations / launches.

    Logic:
      - launch_date = first date with observed sales
      - innovation if the SKU has no positive sales in the previous `lookback_days`
      - keep only SKUs with enough post-launch active days and minimum launch strength
    Returns:
      dict[sku] = {
          'launch_date': ...,
          'dept': ...,
          'launch_mean_qty': ...,
          'post_days': ...
      }
    """
    innovations = {}

    for sku in Y.columns:
        y = Y[sku].astype(float).fillna(0.0)
        a = A[sku].astype(int).fillna(0)

        positive_sales_idx = y[y > 0].index
        if len(positive_sales_idx) == 0:
            continue

        launch_date = positive_sales_idx.min()
        lookback_start = launch_date - timedelta(days=lookback_days)
        pre_window = y.loc[(y.index >= lookback_start) & (y.index < launch_date)]

        # If there were prior positive sales in lookback window, not a launch
        if (pre_window > 0).any():
            continue

        post_window = y.loc[y.index >= launch_date]
        if use_availability_filter:
            post_window = post_window[a.loc[post_window.index] == 1]

        post_days = int((post_window > 0).sum())
        launch_mean_qty = float(post_window.head(28).mean()) if len(post_window) > 0 else 0.0

        if post_days >= min_post_days and launch_mean_qty >= min_launch_sales:
            innovations[sku] = {
                "launch_date": launch_date,
                "dept": sku2dept.get(sku),
                "launch_mean_qty": launch_mean_qty,
                "post_days": post_days
            }

    return innovations

def candidate_victims_of_innovation(
    Y: pd.DataFrame,
    P: pd.DataFrame,
    sku2dept: dict,
    innovation_sku: str,
    launch_date: pd.Timestamp,
    pre_window_days: int = 56,
    post_window_days: int = 56,
    washout_days: int = 0,
    min_pre_days: int = 14,
    min_post_days: int = 14,
    min_post_active_days: int = 7,
    min_relative_drop: float = 0.10
):
    """
    For a given innovation SKU and its launch date, identifies victim SKUs
    in the same department that meet the following criteria:
      - Sufficient data days in both pre and post windows (min_pre_days, min_post_days)
      - At least min_post_active_days days with sales > 0 in the post window
        (filters out discontinued SKUs that drop to 0 for reasons unrelated to the launch)
      - Mean post-launch sales are lower than pre-launch sales (real drop)
      - Relative drop exceeds min_relative_drop threshold
    """
    dept = sku2dept.get(innovation_sku)
    if dept is None:
        return []

    pre_end = launch_date - timedelta(days=1)
    pre_start = pre_end - timedelta(days=pre_window_days - 1)

    post_start = launch_date + timedelta(days=washout_days)
    post_end = post_start + timedelta(days=post_window_days - 1)

    victims = []

    for sku in [s for s in Y.columns if sku2dept.get(s) == dept and s != innovation_sku]:
        y = Y[sku].astype(float).fillna(0.0)
        p_flag = P[sku].astype(bool).reindex(y.index, fill_value=False)

        pre_idx = pd.date_range(pre_start, pre_end, freq="D")
        post_idx = pd.date_range(post_start, post_end, freq="D")

        pre_vals = y.reindex(pre_idx)
        post_vals = y.reindex(post_idx)

        # Exclude the victim's own promotional days from both windows
        pre_vals = pre_vals[~p_flag.reindex(pre_vals.index, fill_value=False)]
        post_vals = post_vals[~p_flag.reindex(post_vals.index, fill_value=False)]

        pre_vals = pre_vals.dropna()
        post_vals = post_vals.dropna()

        # Filter 1: enough data days in both windows
        if len(pre_vals) < min_pre_days or len(post_vals) < min_post_days:
            continue

        # Filter 2: victim must still be active in the post window (discontinuation fix)
        post_active_days = int((post_vals > 0).sum())
        if post_active_days < min_post_active_days:
            continue

        mean_pre = float(pre_vals.mean())
        mean_post = float(post_vals.mean())

        # Filter 3: pre-launch baseline must be positive to compute relative drop
        if mean_pre <= 0:
            continue

        # Filter 4: there must be a real sales drop in the post window
        if mean_post >= mean_pre:
            continue

        relative_drop = (mean_pre - mean_post) / mean_pre

        # Filter 5: drop must exceed the minimum threshold
        if relative_drop < min_relative_drop:
            continue

        victims.append({
            "victim_sku": sku,
            "mean_pre": mean_pre,
            "mean_post": mean_post,
            "relative_drop": float(relative_drop),
            "pre_days": int(len(pre_vals)),
            "post_days": int(len(post_vals)),
            "post_active_days": post_active_days
        })

    return victims



def run_innovation_cannibalization_pipeline(
    df: pd.DataFrame,
    sku_col: str = "analytical_SKU",
    date_col: str = "period_date",
    qty_col: str = "total_qty",
    promo_qty_col: str = "total_qty_promo",
    dept_col: str = "Grp_class_Desc",
    lookback_days: int = 365,
    pre_window_days: int = 56,
    post_window_days: int = 56,
    washout_days: int = 0,
    availability_threshold: float = 0.5,
    weekly_period: int = 7,
    min_launch_sales: float = 3.0,
    min_launch_post_days: int = 14,
    min_pre_days: int = 14,
    min_post_days: int = 14,
    min_relative_drop: float = 0.10,
    threshold_pvalue: float = 0.05,
    temp_series: pd.Series | None = None
):
    """
    End-to-end innovation cannibalization pipeline:
      Step 1: build Y/P/D, decompose signals, build A
      Step 2: detect launched SKUs as innovations
      Step 3: find victim SKUs in same category
      Step 4: quantify impact with CausalImpact / Regression fallback
    """
    Y, P, D, meta = build_core_matrices(
        df,
        sku_col=sku_col,
        date_col=date_col,
        qty_col=qty_col,
        promo_qty_col=promo_qty_col,
        dept_col=dept_col
    )

    trend, seasonal, resid, A = decompose_skus_and_availability(
        Y,
        weekly_period=weekly_period,
        availability_threshold=availability_threshold
    )

    Q_trend, Q_seasonal = decompose_department_signals(
        D,
        weekly_period=weekly_period
    )

    innovations = detect_innovation_launches(
        Y=Y,
        P=P,
        A=A,
        sku2dept=meta["sku_dept"],
        lookback_days=lookback_days,
        min_launch_sales=min_launch_sales,
        min_post_days=min_launch_post_days,
        use_availability_filter=True
    )

    results = []
    extra_controls = {}
    if temp_series is not None:
        extra_controls["temperature"] = temp_series

    for innovation_sku, launch_info in innovations.items():
        dept = launch_info["dept"]
        launch_date = launch_info["launch_date"]

        victims = candidate_victims_of_innovation(
            Y=Y,
            P=P,
            sku2dept=meta["sku_dept"],
            innovation_sku=innovation_sku,
            launch_date=launch_date,
            pre_window_days=pre_window_days,
            post_window_days=post_window_days,
            washout_days=washout_days,
            min_pre_days=min_pre_days,
            min_post_days=min_post_days,
            min_relative_drop=min_relative_drop
        )

        if len(victims) == 0:
            continue

        X_controls = build_control_matrix(
            Q_trend=Q_trend,
            date_index=Y.index,
            dept=dept,
            extra_controls=extra_controls
        )

        intervention_start = launch_date + timedelta(days=washout_days)
        intervention_end = intervention_start + timedelta(days=post_window_days - 1)

        for v in victims:
            y_victim = Y[v["victim_sku"]].astype(float)

            impact = run_causal_effect(
                y=y_victim,
                X=X_controls,
                intervention_start=intervention_start,
                intervention_end=intervention_end
            )

            results.append({
                "innovation_sku": innovation_sku,
                "victim_sku": v["victim_sku"],
                "dept": dept,
                "launch_date": launch_date,
                "intervention_start": intervention_start,
                "intervention_end": intervention_end,
                "innovation_launch_mean_qty": launch_info["launch_mean_qty"],
                "innovation_post_days": launch_info["post_days"],
                "victim_mean_pre": v["mean_pre"],
                "victim_mean_post": v["mean_post"],
                "victim_relative_drop": v["relative_drop"],
                "method": impact["method"],
                "avg_effect": impact["average_effect"],
                "cum_effect": impact["cumulative_effect"],
                "p_value": impact["p_value"],
                "summary_text": impact["summary_text"]
            })

    results_df = pd.DataFrame(results)

    if len(results_df) > 0:
        results_df = results_df.sort_values("p_value", na_position="last")
        results_df = results_df[
            (results_df["avg_effect"] < 0) &
            (results_df["cum_effect"] < 0) &
            (results_df["p_value"] < threshold_pvalue)
        ].copy()

    return {
        "Y": Y,
        "P": P,
        "A": A,
        "D": D,
        "Q_trend": Q_trend,
        "Q_seasonal": Q_seasonal,
        "sku_dept": meta["sku_dept"],
        "innovations": innovations,
        "results": results_df
    }

#### Detect and quantify causal impact

In [ ]:
# =========================
# Step 3: Detecting & Quantifying (Causal Impact)
# =========================

def build_control_matrix(
    Q_trend: pd.DataFrame,
    date_index: pd.DatetimeIndex,
    dept: str,
    extra_controls: dict | None = None
):
    """
    Controls for causal impact:
      - Department trend (Q_trend[dept])
      - Optional external controls (e.g., temperature)
      - Day-of-week dummies
    """
    X = pd.DataFrame(index=date_index)

    if dept in Q_trend.columns:
        X["dept_trend"] = Q_trend[dept]

    if extra_controls is not None:
        for name, series in extra_controls.items():
            X[name] = pd.Series(series).reindex(date_index)

    dow = pd.Series(date_index).dt.dayofweek
    for d in range(7):
        X[f"dow_{d}"] = (dow == d).astype(int).values

    return X.ffill().bfill()


def run_causal_effect(
    y: pd.Series,
    X: pd.DataFrame,
    intervention_start: pd.Timestamp,
    intervention_end: pd.Timestamp
):
    """
    If CausalImpact is available, use it.
    Otherwise use a regression-based fallback on residual shifts.
    """
    df_ci = pd.concat([y.rename("y"), X], axis=1).dropna().sort_index()

    if df_ci.empty:
        return {
            "method": "No data",
            "average_effect": np.nan,
            "cumulative_effect": np.nan,
            "p_value": np.nan,
            "summary_text": "No overlapping data available for causal estimation."
        }

    pre_end = intervention_start - timedelta(days=1)

    if df_ci.index.min() > pre_end or intervention_start > df_ci.index.max():
        return {
            "method": "Insufficient window",
            "average_effect": np.nan,
            "cumulative_effect": np.nan,
            "p_value": np.nan,
            "summary_text": "Insufficient pre/post data for causal estimation."
        }

    pre_period = [df_ci.index.min(), pre_end]
    post_period = [intervention_start, intervention_end]

    if HAS_CAUSALIMPACT:
        try:
            ci = CausalImpact(df_ci, pre_period, post_period)
            causal_inferences = ci.inferences.loc[intervention_start:intervention_end]

            avg_eff = float(causal_inferences["point_effects"].mean()) if "point_effects" in causal_inferences.columns else np.nan

            if "cumulative_effects" in causal_inferences.columns and len(causal_inferences) > 0:
                cum_eff = float(causal_inferences["cumulative_effects"].iloc[-1])
            else:
                cum_eff = np.nan

            try:
                pval = float(ci.p_value)
            except Exception:
                pval = np.nan

            try:
                summary_text = str(ci.summary())
            except Exception:
                summary_text = "CausalImpact completed."

            return {
                "method": "CausalImpact",
                "average_effect": avg_eff,
                "cumulative_effect": cum_eff,
                "p_value": pval,
                "summary_text": summary_text
            }

        except Exception as e:
            pass

    # Regression fallback
    pre_mask = df_ci.index < intervention_start
    post_mask = (df_ci.index >= intervention_start) & (df_ci.index <= intervention_end)

    if pre_mask.sum() < 5 or post_mask.sum() < 3:
        return {
            "method": "Insufficient window",
            "average_effect": np.nan,
            "cumulative_effect": np.nan,
            "p_value": np.nan,
            "summary_text": "Not enough observations in pre or post period."
        }

    reg = LinearRegression()
    reg.fit(df_ci.drop(columns=["y"]), df_ci["y"])
    y_hat = reg.predict(df_ci.drop(columns=["y"]))
    resid = df_ci["y"] - y_hat

    pre_resid = resid[pre_mask]
    post_resid = resid[post_mask]

    effect = float(post_resid.mean() - pre_resid.mean())
    cum_effect = float(effect * post_mask.sum())

    try:
        _, p_val = ttest_ind(pre_resid, post_resid, equal_var=False, nan_policy="omit")
        p_val = float(p_val)
    except Exception:
        p_val = np.nan

    return {
        "method": "Regression-DiD Fallback",
        "average_effect": effect,
        "cumulative_effect": cum_effect,
        "p_value": p_val,
        "summary_text": f"Avg residual shift {effect:.3f} during intervention; p={p_val:.4f}"
    }

#### Orchestration

In [ ]:
# =========================
# Orchestrator
# =========================

def run_innovation_cannibalization_pipeline(
    df: pd.DataFrame,
    temp_series: pd.Series | None = None,
    sku_col: str = "analytical_SKU",
    date_col: str = "period_date",
    qty_col: str = "total_qty",
    promo_qty_col: str = "total_qty_promo",
    dept_col: str = "Grp_class_Desc",
    lookback_days: int = 365,
    pre_window_days: int = 56,
    post_window_days: int = 56,
    washout_days: int = 0,
    availability_threshold: float = 0.5,
    weekly_period: int = 7,
    min_launch_sales: float = 3.0,
    min_launch_post_days: int = 14,
    min_pre_days: int = 14,
    min_post_days: int = 14,
    min_post_active_days = 7,
    min_relative_drop: float = 0.10,
    threshold_pvalue: float = 0.05
):
    """
    End-to-end pipeline:
      Step 1: build Y, P, D; decompose Y and D; compute availability A and Q signals
      Step 2: detect innovation launches
      Step 3: find victims after launch and run causal impact / regression DiD
    Returns:
      dict with matrices, decompositions, detected innovations, and results DataFrame.
    """

    # Step 1
    Y, P, D, meta = build_core_matrices(
        df,
        sku_col=sku_col,
        date_col=date_col,
        qty_col=qty_col,
        promo_qty_col=promo_qty_col,
        dept_col=dept_col
    )

    trend, seasonal, resid, A = decompose_skus_and_availability(
        Y,
        weekly_period=weekly_period,
        availability_threshold=availability_threshold
    )

    Q_trend, Q_seasonal = decompose_department_signals(
        D,
        weekly_period=weekly_period
    )

    # Step 2
    innovations = detect_innovation_launches(
        Y=Y,
        P=P,
        A=A,
        sku2dept=meta["sku_dept"],
        lookback_days=lookback_days,
        min_launch_sales=min_launch_sales,
        min_post_days=min_launch_post_days,
        use_availability_filter=True
    )

    # Step 3
    results = []
    extra_controls = {}
    if temp_series is not None:
        extra_controls["temperature"] = temp_series

    for innovation_sku, launch_info in innovations.items():
        dept = launch_info["dept"]
        launch_date = launch_info["launch_date"]

        victims = candidate_victims_of_innovation(
            Y=Y,
            P=P,
            sku2dept=meta["sku_dept"],
            innovation_sku=innovation_sku,
            launch_date=launch_date,
            pre_window_days=pre_window_days,
            post_window_days=post_window_days,
            washout_days=washout_days,
            min_pre_days=min_pre_days,
            min_post_days=min_post_days,
            min_relative_drop=min_relative_drop,
            min_post_active_days = min_post_active_days
        )

        if len(victims) == 0:
            continue

        X_controls = build_control_matrix(
            Q_trend=Q_trend,
            date_index=Y.index,
            dept=dept,
            extra_controls=extra_controls
        )

        intervention_start = launch_date + timedelta(days=washout_days)
        intervention_end = intervention_start + timedelta(days=post_window_days - 1)

        for v in victims:
            y_victim = Y[v["victim_sku"]].astype(float)

            impact = run_causal_effect(
                y=y_victim,
                X=X_controls,
                intervention_start=intervention_start,
                intervention_end=intervention_end
            )

            results.append({
                "innovation_sku": innovation_sku,
                "victim_sku": v["victim_sku"],
                "dept": dept,
                "launch_date": launch_date,
                "intervention_start": intervention_start,
                "intervention_end": intervention_end,
                "innovation_launch_mean_qty": launch_info["launch_mean_qty"],
                "innovation_post_days": launch_info["post_days"],
                "victim_mean_pre": v["mean_pre"],
                "victim_mean_post": v["mean_post"],
                "victim_relative_drop": v["relative_drop"],
                "method": impact["method"],
                "avg_effect": impact["average_effect"],
                "cum_effect": impact["cumulative_effect"],
                "p_value": impact["p_value"],
                "summary_text": impact["summary_text"]
            })

    results_df = pd.DataFrame(results)

    if len(results_df) > 0:
        results_df = results_df.sort_values("p_value", na_position="last")
        results_df = results_df[
            (results_df["avg_effect"] < 0) &
            (results_df["cum_effect"] < 0) &
            (results_df["p_value"] < threshold_pvalue)
        ].copy()

    return {
        "Y": Y,
        "P": P,
        "A": A,
        "D": D,
        "Q_trend": Q_trend,
        "Q_seasonal": Q_seasonal,
        "sku_dept": meta["sku_dept"],
        "innovations": innovations,
        "results": results_df
    }

#### Manually test the pipeline

Test on Top 10 stores by sales in France

## RUN PIPELINE FOR 5 STORES


In [ ]:
# =========================
# Manual run by store
# =========================

all_results = []

# Parameters
sku_col = "analytical_SKU"
date_col = "period_date"
qty_col = "total_qty"
promo_qty_col = "total_qty_promo"
dept_col = "Grp_class_Desc"

lookback_days = 365
pre_window_days = 56
post_window_days = 56
washout_days = 0
availability_threshold = 0.5
weekly_period = 7
min_launch_sales = 3.0
min_launch_post_days = 14
min_pre_days = 14
min_post_days = 14
min_relative_drop = 0.10
threshold_pvalue = 0.05
min_post_active_days = 7
temp_series = None

for store_id, df_store in dfPdcan_top20.groupby("store_key"):
    print(f"Processing store_key = {store_id}")

    output = run_innovation_cannibalization_pipeline(
        df=df_store,
        sku_col=sku_col,
        date_col=date_col,
        qty_col=qty_col,
        promo_qty_col=promo_qty_col,
        dept_col=dept_col,
        lookback_days=lookback_days,
        pre_window_days=pre_window_days,
        post_window_days=post_window_days,
        washout_days=washout_days,
        availability_threshold=availability_threshold,
        weekly_period=weekly_period,
        min_launch_sales=min_launch_sales,
        min_launch_post_days=min_launch_post_days,
        min_pre_days=min_pre_days,
        min_post_days=min_post_days,
        min_relative_drop=min_relative_drop,
        threshold_pvalue=threshold_pvalue,
        temp_series=temp_series,
        min_post_active_days=min_post_active_days,
    )

    results_df = output["results"]

    if len(results_df) > 0:
        results_df["store_key"] = store_id
        results_df["store_description"] = df_store["store_description"].iloc[0]
        all_results.append(results_df)

if len(all_results) > 0:
    df_final = pd.concat(all_results, ignore_index=True)
    df_final = df_final.sort_values(["p_value", "avg_effect"], ascending=[True, True])
else:
    df_final = pd.DataFrame(columns=[
        "innovation_sku", "victim_sku", "dept", "launch_date",
        "intervention_start", "intervention_end",
        "innovation_launch_mean_qty", "innovation_post_days",
        "victim_mean_pre", "victim_mean_post", "victim_relative_drop",
        "method", "avg_effect", "cum_effect", "p_value", "summary_text",
        "store_key", "store_description"
    ])

display(df_final)

In [ ]:
df_final.groupby("store_key")["innovation_sku"].nunique()

In [ ]:
display(df_final)

In [ ]:
df_final = df_final[ (df_final["avg_effect"] < 0) & (df_final["cum_effect"] < 0) & (df_final["p_value"] < threshold_pvalue) ].copy()

In [ ]:
display(df_final)

In [ ]:
df_danone = df_final[(df_final["innovation_sku"].str.upper().str.contains("DANONE")) | (df_final["victim_sku"].str.upper().str.contains("DANONE"))].copy()

In [ ]:
display(df_danone)

In [ ]:
def extract_supplier_brand(sku):
    parts = sku.split("_")
    if len(parts) >= 2:
        supplier = parts[0]
        brand = parts[1]
    else:
        supplier = None
        brand = None
    return supplier, brand

In [ ]:
df_danone["innovation_supplier"] = df_danone["innovation_sku"].apply(lambda x: extract_supplier_brand(x)[0])
df_danone["innovation_brand"] = df_danone["innovation_sku"].apply(lambda x: extract_supplier_brand(x)[1])

df_danone["victim_supplier"] = df_danone["victim_sku"].apply(lambda x: extract_supplier_brand(x)[0])
df_danone["victim_brand"] = df_danone["victim_sku"].apply(lambda x: extract_supplier_brand(x)[1])

In [ ]:
display(df_danone)

In [ ]:
# Adding two new columns to help measure the efect of cannibalization:
# abs_loss_per_day: Tells us the victim lost X amount of units per day in average
df_danone["abs_loss_per_day"] = df_danone["victim_mean_pre"] - df_danone["victim_mean_post"]

# total units lost: In the whole post_launch window, the total loss was Y amount of units
df_danone["total_units_lost"] = df_danone["abs_loss_per_day"] * (
    (df_danone["intervention_end"] - df_danone["intervention_start"]).dt.days + 1
)

In [ ]:
display(df_danone)

In [ ]:
# =============================================
# Brand-level cannibalization analysis
# =============================================

# Step 1: Aggregate by brand pair across all stores
df_brand = (
    df_danone
    .groupby(["innovation_brand", "victim_brand"])
    .agg(
        n_pairs              = ("victim_sku",          "count"),
        n_stores             = ("store_key",           "nunique"),
        total_units_lost     = ("total_units_lost",    "sum"),
        avg_cum_effect       = ("cum_effect",          "mean"),
        avg_relative_drop    = ("victim_relative_drop","mean"),
        avg_pvalue           = ("p_value",             "mean")
    )
    .reset_index()
    .sort_values("total_units_lost", ascending=False)
)

display(df_brand)

In [ ]:
# Step 2: Focus on Danone as victim (how much did other brands cannibalize Danone?)
danone_as_victim = df_brand[df_brand["victim_brand"].isin(
    df_danone[df_danone["victim_supplier"] == "DANONE"]["victim_brand"].unique()
)].sort_values("total_units_lost", ascending=False)

display(danone_as_victim)

In [ ]:
# Step 3: Focus on Danone as cannibal (how much did Danone cannibalize others?)
danone_as_cannibal = df_brand[df_brand["innovation_brand"].isin(
    df_danone[df_danone["innovation_supplier"] == "DANONE"]["innovation_brand"].unique()
)].sort_values("total_units_lost", ascending=False)

display(danone_as_cannibal)